In [1]:
from pathlib import Path

In [2]:
data_dir = Path("/home/rooneyish/Documents/Pulse_SLM/data")
train_file = data_dir / "train.txt"
val_file = data_dir / "val.txt"

In [3]:
train_txt = open(train_file).read()
val_txt = open(val_file).read()

In [4]:
import regex as re

def clean_txt(text):
    # Remove the backslashes escaping the single quotes (\'s -> 's)
    text = re.sub(r"[^\p{Latin}\p{P}\p{N}\p{Z}]", " ", text)
    text = text.replace("\\'", "'")
    
    # Fix the weird spacing around the text formatting
    text = re.sub(r"\s+'\s*([sn])", r"'\1", text)   # fixes ' s or ' n
    text = re.sub(r"={1,}\s*(.*?)\s*={1,}", r"\1", text) # removes headers
    text = re.sub(r"\s+([,.:;?!])", r"\1", text)    # fixes punctuation spacing
    text = re.sub(r"\s*@\s*-\s*@\s*", "-", text)    # fixes @-@ hyphens
    text = re.sub(r"\s+", " ", text).strip()        # collapses extra spaces
    text = text.replace("<unk>", "")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    text = text.lower()

    
    return text

In [5]:
clean_train_txt = clean_txt(train_txt)
clean_val_txt = clean_txt(val_txt)

In [6]:
len(clean_train_txt), len(clean_val_txt)

(10514596, 1103205)

In [7]:
full_txt = clean_train_txt + clean_val_txt

In [8]:
len(full_txt)

11617801

In [9]:
import nltk.data

nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /home/rooneyish/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rooneyish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from nltk.tokenize import sent_tokenize

sent_tokens = sent_tokenize(full_txt)

In [11]:
sent_tokens[:5]

['valkyria chronicles iii senjō no valkyria 3: unrecorded chronicles ( japanese: 3, lit.',
 'valkyria of the battlefield 3 ), commonly referred to as valkyria chronicles iii outside japan, is a tactical role-playing video game developed by sega and media.vision for the playstation portable.',
 'released in january 2011 in japan, it is the third game in the valkyria series.',
 'employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " nameless ", a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit " calamaty raven ".',
 'the game began development in 2010, carrying over a large portion of the work done on valkyria chronicles ii.']

In [12]:
sent_w_SE = []
for each in sent_tokens:
    each = ''.join('<S> '+ each + ' <E>')
    sent_w_SE.append(each)

In [13]:
sent_w_SE[:5]

['<S> valkyria chronicles iii senjō no valkyria 3: unrecorded chronicles ( japanese: 3, lit. <E>',
 '<S> valkyria of the battlefield 3 ), commonly referred to as valkyria chronicles iii outside japan, is a tactical role-playing video game developed by sega and media.vision for the playstation portable. <E>',
 '<S> released in january 2011 in japan, it is the third game in the valkyria series. <E>',
 '<S> employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the " nameless ", a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit " calamaty raven ". <E>',
 '<S> the game began development in 2010, carrying over a large portion of the work done on valkyria chronicles ii. <E>']

In [14]:
tokens = []
for sentence in sent_w_SE:
    word = sentence.split()
    tokens.append(word)

In [15]:
len(tokens)

80169

In [16]:
corpus = {}
all_tokens = []
for each in tokens:
    for i in range(len(each)):
        all_tokens.append(each[i])

for each in all_tokens:
    if each in corpus:
        corpus[each] += 1
    else:
        corpus[each] = 1

corpus = {k:v for k, v in sorted(corpus.items(), key = lambda item: item[1], reverse = True)}
del corpus['<S>']
del corpus['<E>']
len(corpus)

120377

In [17]:
start_token = "<S>"
end_token = "<E>"
start_id = 1
end_id = 2

unique_words = list(corpus.keys())

In [18]:
stoi = {start_token: start_id, end_token: end_id}
itos = {start_id: start_token, end_id: end_token}

In [19]:
for idx, word in enumerate(unique_words):
    target_id = idx + 3
    stoi[word] = target_id
    itos[target_id] = word

In [20]:
len(stoi), len(itos)

(120379, 120379)

In [21]:
import torch

In [22]:
tokenized_sentence = [[stoi[token] for token in sentence if token in stoi] for sentence in tokens]

In [23]:
X_data = []
Y_data = []

In [24]:
max(map(len, tokenized_sentence))

568

In [25]:
tokenized_sentence[:2]

[[1,
  4010,
  4011,
  1169,
  23810,
  98,
  4010,
  4012,
  41854,
  4011,
  17,
  21054,
  1448,
  12175,
  2],
 [1,
  4010,
  4,
  3,
  8487,
  112,
  58,
  1667,
  819,
  7,
  12,
  4010,
  4011,
  1169,
  533,
  2888,
  18,
  8,
  5455,
  9998,
  234,
  100,
  427,
  16,
  7421,
  5,
  33046,
  14,
  3,
  1610,
  33047,
  2]]

In [26]:
sequence = []
for sentence in tokenized_sentence:
    for i in range(1, len(sentence)):
        sequence.append(sentence[:i+1])

In [27]:
sequence[:20]

[[1, 4010],
 [1, 4010, 4011],
 [1, 4010, 4011, 1169],
 [1, 4010, 4011, 1169, 23810],
 [1, 4010, 4011, 1169, 23810, 98],
 [1, 4010, 4011, 1169, 23810, 98, 4010],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012, 41854],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012, 41854, 4011],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012, 41854, 4011, 17],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012, 41854, 4011, 17, 21054],
 [1, 4010, 4011, 1169, 23810, 98, 4010, 4012, 41854, 4011, 17, 21054, 1448],
 [1,
  4010,
  4011,
  1169,
  23810,
  98,
  4010,
  4012,
  41854,
  4011,
  17,
  21054,
  1448,
  12175],
 [1,
  4010,
  4011,
  1169,
  23810,
  98,
  4010,
  4012,
  41854,
  4011,
  17,
  21054,
  1448,
  12175,
  2],
 [1, 4010],
 [1, 4010, 4],
 [1, 4010, 4, 3],
 [1, 4010, 4, 3, 8487],
 [1, 4010, 4, 3, 8487, 112],
 [1, 4010, 4, 3, 8487, 112, 58]]

In [28]:
max_len = max(map(len, sequence))
max_len

568

In [29]:
from torch.nn.utils.rnn import pad_sequence
tensor_seq = [torch.tensor(seq) for seq in sequence]
padded_sequence = pad_sequence(tensor_seq, batch_first=True, padding_side='left', padding_value=0)

In [30]:
padded_sequence[100]

tensor([    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [31]:
len(padded_sequence[100])

568

In [32]:
X = padded_sequence[:, :-1]
y = padded_sequence[:,-1]

In [33]:
X[100]

tensor([    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [34]:
y[100]

tensor(149)

In [35]:
num_samples = X.shape[0]
num_samples

2059355

In [36]:
split_idx_1= int(num_samples * 0.8)
split_idx_2= int(num_samples * 0.9)

In [37]:
X_train, y_train = X[:split_idx_1], y[:split_idx_1]
X_val, y_val = X[split_idx_1:split_idx_2], y[split_idx_1:split_idx_2]
X_test, y_test = X[split_idx_2:], y[split_idx_2:]

In [38]:
len(X_train), len(y_train)

(1647484, 1647484)

In [39]:
len(X_val), len(y_val)

(205935, 205935)

In [40]:
len(X_test), len(y_test)

(205936, 205936)